In [1]:
import os
import imageio_ffmpeg
import whisper
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

# Configure ffmpeg path so Whisper can find it
ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
os.environ["PATH"] += os.pathsep + os.path.dirname(ffmpeg_exe)
print("Configured FFmpeg at:", ffmpeg_exe)

DATA_DIR = "Dataset_Final"
TRAIN_AUDIO_DIR = os.path.join(DATA_DIR, "train")
TEST_AUDIO_DIR = os.path.join(DATA_DIR, "test")


# Load lightweight open-source whisper model on CPU
print("Loading Whisper model...")
asr_model = whisper.load_model("tiny.en", device="cpu")
print("Model loaded successfully!")

Configured FFmpeg at: c:\Users\divya\OneDrive\Desktop\Grammar_Scoring_engine\venv\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe
Loading Whisper model...
Model loaded successfully!


In [2]:
import os
import shutil
import imageio_ffmpeg

# Get the binary and directory
real_ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
ffmpeg_dir = os.path.dirname(real_ffmpeg)
target_ffmpeg = os.path.join(ffmpeg_dir, "ffmpeg.exe")

# Copy the binary to ffmpeg.exe if not already present
if not os.path.exists(target_ffmpeg):
    shutil.copyfile(real_ffmpeg, target_ffmpeg)
    print("Created ffmpeg.exe at:", target_ffmpeg)
else:
    print("ffmpeg.exe already exists at:", target_ffmpeg)

# Prepend ffmpeg directory to the beginning of PATH
os.environ["PATH"] = ffmpeg_dir + os.pathsep + os.environ["PATH"]
print("PATH updated successfully!")

Created ffmpeg.exe at: c:\Users\divya\OneDrive\Desktop\Grammar_Scoring_engine\venv\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg.exe
PATH updated successfully!


In [3]:
def extract_audio_and_transcript_features(audio_path, model):
    """
    Extracts:
    1. Whisper transcript (captures grammatical phrasing, syntax, and pauses)
    2. Audio duration (seconds)
    3. Word count and speech rate (words per second)
    4. Silence / pause ratio using librosa RMS energy
    """
    # 1. Transcribe
    result = model.transcribe(audio_path, fp16=False)
    transcript = result["text"].strip()
    
    # 2. Audio duration & silence ratio via librosa
    y, sr = librosa.load(audio_path, sr=16000)
    duration = librosa.get_duration(y=y, sr=sr)
    
    # Estimate speech vs pause (energy-based voice activity detection)
    intervals = librosa.effects.split(y, top_db=25)
    voiced_duration = sum([(end - start) / sr for start, end in intervals])
    silence_ratio = max(0.0, (duration - voiced_duration) / duration) if duration > 0 else 0.0
    
    # 3. Simple speech rate metrics
    word_count = len(transcript.split())
    words_per_sec = word_count / duration if duration > 0 else 0.0
    
    return {
        "transcript": transcript,
        "duration": duration,
        "word_count": word_count,
        "words_per_sec": words_per_sec,
        "silence_ratio": silence_ratio
    }

In [5]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
sample_audio_path = os.path.join(TRAIN_AUDIO_DIR, train_df.iloc[0]["filename"])
print(f"Testing extraction on: {sample_audio_path}")

sample_result = extract_audio_and_transcript_features(sample_audio_path, asr_model)
print("\nExtraction Success!")
for k, v in sample_result.items():
    print(f"  {k}: {v}")
    

Testing extraction on: Dataset_Final\train\audio_192.wav

Extraction Success!
  transcript: The gallon is one of the temperature of the mouth Rico. Different temperature of the oven is canceled and cooled. Every weekend and moremed is donate to food and food centre. The link is already enjoyed this thing. It has to be like, it's more like a game of pay. Once it's smooth and the other thing will have to push and the mirror go around, which was fine. And most of them live for... Once and perhaps white or dust, you'll be smooth. Some of the key blights you do sound cautious. Most players will hear the sound of streaming children lasting tears.
  duration: 60.508375
  word_count: 104
  words_per_sec: 1.7187703355114063
  silence_ratio: 0.46374365532044093


In [6]:
# Process Train Data
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_records = []

print("Extracting features for Training Set...")
for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):
    audio_path = os.path.join(TRAIN_AUDIO_DIR, row["filename"])
    feats = extract_audio_and_transcript_features(audio_path, asr_model)
    feats["filename"] = row["filename"]
    feats["label"] = row["label"]
    train_records.append(feats)

train_features_df = pd.DataFrame(train_records)
train_features_df.to_csv("train_features.csv", index=False)
print("Saved train_features.csv")

# Process Test Data
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_records = []

print("Extracting features for Test Set...")
for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    audio_path = os.path.join(TEST_AUDIO_DIR, row["filename"])
    feats = extract_audio_and_transcript_features(audio_path, asr_model)
    feats["filename"] = row["filename"]
    test_records.append(feats)

test_features_df = pd.DataFrame(test_records)
test_features_df.to_csv("test_features.csv", index=False)
print("Saved test_features.csv")

Extracting features for Training Set...


100%|██████████| 769/769 [49:16<00:00,  3.84s/it]  


Saved train_features.csv
Extracting features for Test Set...


100%|██████████| 216/216 [10:40<00:00,  2.96s/it]

Saved test_features.csv
